# ResNet50 mediante Transfer Learning

Clasificación de PICADURA y RUSSET en palta Hass.

Se conserva este notebook como punto de entrada al entrenamiento. Se completan
las funciones existentes para comparar un **baseline B0** con una o dos variantes
**de los datos**, sin cambiar de arquitectura ni reorganizar el proyecto.

| Condición | Alto x ancho | Volteo horizontal en train |
|---|---|---|
| B0: referencia | 256 x 512 | No |
| V1: mayor resolución | 384 x 768 | No |
| V2: aumento de datos (bloqueada) | 256 x 512 | No ejecutada |

**Primera apertura:** seleccionar el kernel `palta` y ejecutar todas las celdas con
los controles de la sección 8 en `False`. Así se cargan funciones y se muestra
la propuesta de exclusiones, sin descargar pesos, entrenar ni evaluar test.
Después se ejecuta una condición por vez siguiendo esa sección.

Los resultados EDA y Feature Engineering se conservan en sus carpetas. El informe
está en `2. informe/Informe.pdf` y su versión editable en `Informe.docx`.
B0 y V1 se ejecutan con TIFF locales. La sección 9 consulta las métricas reales guardadas. V2 está bloqueada: el responsable confirmó que debe conservarse el orden horizontal de las capturas, que el volteo invertiría.

## 1. Configuración

Cada archivo representa una palta con tres franjas de cámaras. El pipeline conserva
la imagen completa y ajusta su tamaño a **256 × 512 píxeles (alto × ancho)** con relleno,
sin recorte ni deformación. Es una propuesta inicial; la conservación de defectos
pequeños debe revisarse con las imágenes reales antes de fijar esta resolución.

`utilidades_imagenes.py` comparte la lectura RGB, la compatibilidad TIFF y el ajuste
de tamaño con Feature Engineering. Los tensores llegan al modelo en RGB, `float32`,
escala 0–255. El preprocesamiento específico de ResNet50 se aplica dentro del modelo.


In [1]:
from pathlib import Path, PureWindowsPath
import sys
import hashlib
import json
import random
import time
import platform
import importlib.metadata
from IPython.display import display

import numpy as np
import pandas as pd

# Encontrar la raíz desde el notebook o desde el repositorio.
RAIZ = next((ruta for ruta in [Path.cwd(), *Path.cwd().parents]
             if (ruta / "README.md").is_file()
             and (ruta / "utilidades_imagenes.py").is_file()), None)
if RAIZ is None:
    raise FileNotFoundError("Abra el notebook desde la carpeta del proyecto.")
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from utilidades_imagenes import (
    encontrar_raiz, resolver_dataset, listar_imagenes, crear_dataset_tf,
)

RAIZ = encontrar_raiz()
RUTA_DATASET_RELATIVA = "../DATASET_PALTA"
import os
os.environ.setdefault("KERAS_HOME", str(RAIZ / ".cache" / "keras"))
SALIDA = RAIZ / "7. resultados" / "entrenamiento"
SEMILLA = 42
TAMANO_ENTRADA = (256, 512)
TAMANO_LOTE = 8
INSPECCIONAR_LOTE = False
VOLTEO_VALIDADO = False  # El responsable confirmó que debe conservarse el orden horizontal.

random.seed(SEMILLA)
np.random.seed(SEMILLA)
EXPERIMENTOS = {
    "B0": {"tamano": (256, 512), "volteo_horizontal": False},
    "V1": {"tamano": (384, 768), "volteo_horizontal": False},
    "V2": {"tamano": (256, 512), "volteo_horizontal": True},
}
print("Funciones de ResNet50. El entrenamiento se activa en la seccion 8.")

Funciones de ResNet50. El entrenamiento se activa en la seccion 8.


## 2. Datos de entrenamiento y validación

Las etiquetas se definen una vez a partir de las clases de train y se reutilizan en
validation y, al finalizar el experimento, en test. La palta completa sigue siendo
la unidad de observación. No se separan las franjas en muestras independientes.

Los archivos originales se leen desde sus particiones actuales. El notebook no
mueve, renombra ni sobrescribe imágenes.


In [2]:
def preparar_datos(excluidos=()):
    dataset = resolver_dataset(RAIZ, RUTA_DATASET_RELATIVA)
    tabla_train = listar_imagenes(dataset, "train")
    if tabla_train.empty:
        raise ValueError("Train no contiene imagenes.")
    clases = sorted(tabla_train["clase"].unique().tolist())
    if clases != ["PICADURA", "RUSSET"]:
        raise ValueError("Se esperaban las clases PICADURA y RUSSET.")
    tabla_val = listar_imagenes(dataset, "validation", clases=clases)
    tabla_train = tabla_train.loc[~tabla_train["archivo"].isin(excluidos)].reset_index(drop=True)
    tabla_val = tabla_val.loc[~tabla_val["archivo"].isin(excluidos)].reset_index(drop=True)
    for nombre, tabla in (("train", tabla_train), ("validation", tabla_val)):
        if tabla.empty or set(tabla["clase"]) != set(clases):
            raise ValueError(f"{nombre} debe conservar ambas clases despues de excluir duplicados.")
    return dataset, tabla_train, tabla_val, clases


def calcular_pesos(tabla_train, clases):
    from sklearn.utils.class_weight import compute_class_weight

    etiquetas = tabla_train["etiqueta"].to_numpy(dtype=np.int32)
    valores = np.arange(len(clases), dtype=np.int32)
    pesos = compute_class_weight("balanced", classes=valores, y=etiquetas)
    return {int(clase): float(peso) for clase, peso in zip(valores, pesos)}

### Inspección opcional de un lote

Al activar `INSPECCIONAR_LOTE` se carga un lote de train, sin aumento ni
preprocesamiento de ResNet50. Sirve para comprobar forma, rango y etiquetas.
No construye una red ni consulta imágenes de test.


In [3]:
if INSPECCIONAR_LOTE:
    import matplotlib.pyplot as plt

    dataset, tabla_train, tabla_val, clases = preparar_datos()
    datos_revision = crear_dataset_tf(
        tabla_train, dataset, tamano=TAMANO_ENTRADA,
        tamano_lote=TAMANO_LOTE, entrenamiento=False, semilla=SEMILLA,
    )
    imagenes, etiquetas = next(iter(datos_revision))
    print("Forma:", tuple(imagenes.shape), "Tipo:", imagenes.dtype.name)
    print("Rango RGB:", float(np.min(imagenes)), float(np.max(imagenes)))
    print("Clases:", {indice: clase for indice, clase in enumerate(clases)})
    plt.figure(figsize=(10, 5))
    plt.imshow(imagenes[0].numpy().astype(np.uint8))
    plt.title(clases[int(etiquetas[0])])
    plt.axis("off")
    plt.show()
else:
    print("Inspección de lote desactivada.")


Inspección de lote desactivada.


## 3. Condición previa: auditoría de datos

El EDA registra **dos pares exactos entre particiones**, uno con etiquetas distintas.
El CSV contiene cuatro filas porque cada par se detectó por bytes y por píxeles;
no son cuatro pares diferentes.

Se propone excluir **todos los archivos de esos pares de la lista experimental**.
No se borra, mueve, renombra, reetiqueta ni redistribuye ningún TIFF. La misma
lista se aplica a B0, V1 y V2. Las exclusiones fueron autorizadas para esta ejecución y se aplicaron tras verificar los 829 hashes; la sección 8 conserva el control explícito para futuras repeticiones.

La auditoría comprueba la firma del inventario, las rutas y los SHA-256 locales.
También verifica que las exclusiones no dejen duplicados exactos entre particiones.
Si un archivo cambió o faltan reportes, se detiene: no basta con activar un control.
Test se consulta solo para integridad; no se predice ni se utiliza para elegir variantes.

**Límite:** esta verificación no prueba independencia por fruto o lote. La
comparación es exploratoria por archivo completo hasta disponer de esa trazabilidad.

In [4]:
def hash_archivo(ruta):
    calculo = hashlib.sha256()
    with Path(ruta).open("rb") as archivo:
        for bloque in iter(lambda: archivo.read(1024 * 1024), b""):
            calculo.update(bloque)
    return calculo.hexdigest()


def firma_tabla(tabla):
    texto = "\n".join(tabla["archivo"] + ":" + tabla["sha256"])
    return hashlib.sha256(texto.encode()).hexdigest()


def leer_auditoria():
    carpeta = RAIZ / "7. resultados" / "eda"
    resumen = json.loads((carpeta / "eda_resumen.json").read_text(encoding="utf-8-sig"))
    inventario = pd.read_csv(carpeta / "eda_inventario.csv", encoding="utf-8-sig", keep_default_na=False)
    duplicados = pd.read_csv(carpeta / "eda_duplicados.csv", encoding="utf-8-sig", keep_default_na=False)
    columnas = {"archivo", "sha256", "sha_pixeles", "particion", "clase", "imagen", "error"}
    if not columnas.issubset(inventario.columns):
        raise ValueError("El inventario no contiene la auditoria necesaria.")
    if inventario.empty or inventario["archivo"].duplicated().any():
        raise ValueError("El inventario esta vacio o contiene rutas repetidas.")
    if len(inventario) != resumen["archivos"] or len(inventario) != resumen["imagenes"]:
        raise ValueError("Los conteos no coinciden. Revise el EDA.")
    if resumen["corruptas"] or inventario["error"].ne("").any():
        raise ValueError("El EDA registra errores de lectura.")
    if not inventario["imagen"].astype(str).str.lower().isin(["true", "1"]).all():
        raise ValueError("El inventario incluye archivos no reconocidos como imagenes.")
    if firma_tabla(inventario) != resumen["sha256_dataset"]:
        raise ValueError("La firma del inventario no coincide con el resumen.")
    for campo in ("sha256", "sha_pixeles"):
        if not inventario[campo].astype(str).str.fullmatch(r"[0-9a-f]{64}").all():
            raise ValueError(f"Hashes incompletos en {campo}.")
    for registro in inventario.itertuples(index=False):
        ruta = Path(registro.archivo)
        win = PureWindowsPath(registro.archivo)
        if (ruta.is_absolute() or win.drive or win.root or ".." in ruta.parts
                or "\\" in registro.archivo or len(ruta.parts) < 3):
            raise ValueError("El inventario debe usar rutas relativas dentro del dataset.")
        if ruta.parts[:2] != (registro.particion, registro.clase):
            raise ValueError(f"Ruta y etiqueta incoherentes: {registro.archivo}")
        if registro.particion not in {"train", "validation", "test"}:
            raise ValueError("Particion no reconocida.")
        if registro.clase not in {"PICADURA", "RUSSET"}:
            raise ValueError("Clase no reconocida.")

    requeridas = {"tipo", "archivo", "archivo_coincidente", "particion",
                  "particion_coincidente", "clase", "clase_coincidente"}
    if not requeridas.issubset(duplicados.columns):
        raise ValueError("El reporte de duplicados esta incompleto.")
    exactos = duplicados[duplicados["tipo"].isin(["bytes_exactos", "pixeles_exactos"])]
    indexado = inventario.set_index("archivo")
    pares_fuga, pares_conflicto, excluidos = set(), set(), set()
    for fila in exactos.itertuples(index=False):
        a, b = fila.archivo, fila.archivo_coincidente
        if a == b or a not in indexado.index or b not in indexado.index:
            raise ValueError("El reporte de duplicados contiene una ruta invalida.")
        ra, rb = indexado.loc[a], indexado.loc[b]
        if (ra.particion, rb.particion, ra.clase, rb.clase) != (
                fila.particion, fila.particion_coincidente, fila.clase, fila.clase_coincidente):
            raise ValueError("Duplicados e inventario no coinciden.")
        clave = "sha256" if fila.tipo == "bytes_exactos" else "sha_pixeles"
        if ra[clave] != rb[clave]:
            raise ValueError("El par reportado no coincide con los hashes del inventario.")
        par = tuple(sorted([a, b]))
        fuga, conflicto = ra.particion != rb.particion, ra.clase != rb.clase
        if fuga:
            pares_fuga.add(par)
        if conflicto:
            pares_conflicto.add(par)
        if fuga or conflicto:
            excluidos.update([a, b])
    if (len(pares_fuga) != resumen["pares_exactos_entre_particiones"]
            or len(pares_conflicto) != resumen["pares_exactos_etiquetas_distintas"]):
        raise ValueError("Los conteos de pares no coinciden con el resumen.")

    elegibles = inventario.loc[~inventario["archivo"].isin(excluidos)].copy()
    for clave in ("sha256", "sha_pixeles"):
        grupos = elegibles.groupby(clave)[["particion", "clase"]].nunique()
        if (grupos > 1).any().any():
            raise ValueError("Persisten duplicados entre particiones o clases. Revise el EDA.")
    return resumen, inventario, sorted(excluidos)


def verificar_auditoria(dataset, aprobar_exclusiones=False):
    resumen, inventario, excluidos = leer_auditoria()
    dataset = Path(dataset).resolve()
    rutas = set(inventario["archivo"])
    locales = {p.relative_to(dataset).as_posix() for p in dataset.rglob("*") if p.is_file()}
    if locales != rutas:
        raise RuntimeError("El contenido del dataset cambio. Revise las rutas y actualice el EDA.")
    total = len(inventario)
    for numero, registro in enumerate(inventario.itertuples(index=False), 1):
        ruta = (dataset / registro.archivo).resolve()
        if not ruta.is_relative_to(dataset):
            raise ValueError("Un enlace apunta fuera del dataset.")
        if hash_archivo(ruta) != registro.sha256:
            raise RuntimeError(f"Archivo distinto al auditado: {registro.archivo}")
        if numero % 100 == 0 or numero == total:
            print(f"Integridad: {numero}/{total}")
    if excluidos and not aprobar_exclusiones:
        raise RuntimeError(
            "Integridad correcta; falta aprobar las exclusiones mostradas en la seccion 8. "
            "No se entreno ni se modificaron imagenes."
        )
    elegibles = inventario.loc[~inventario["archivo"].isin(excluidos)].sort_values("archivo")
    return {
        "sha256_dataset": resumen["sha256_dataset"],
        "sha256_manifiesto": firma_tabla(elegibles),
        "exclusiones": excluidos,
        "archivos_originales": int(len(inventario)),
        "archivos_elegibles": int(len(elegibles)),
        "alcance": "Integridad exacta; independencia por fruto/lote pendiente de trazabilidad.",
    }

## 4. Arquitectura

**Imagen RGB -> volteo opcional en train -> preprocesamiento ResNet50 -> extractor
congelado -> Dropout -> clasificación.**

Se reutilizan el extractor, la cabeza y el optimizador del notebook original. B0
no aplica augmentation; V1 solo cambia la resolución; V2 proponía el volteo, pero está bloqueada por incompatibilidad semántica.
Los tres entrenan una cabeza nueva desde la misma semilla. No se continúan los
pesos aprendidos en B0 al ejecutar una variante.

`preprocess_input` transforma RGB a BGR y centra los canales una sola vez, sin
escalar a [0,1]. ResNet50 queda con `trainable=False` y `training=False`; se ajusta
la cabeza de clasificación, no todas las capas. Las funciones usan los pesos
ImageNet solo al iniciar el experimento. [Keras: ResNet](https://keras.io/api/applications/resnet/resnet_models/)
y [Transfer Learning](https://keras.io/guides/transfer_learning/).

In [5]:
def construir_modelo(numero_clases, tamano=TAMANO_ENTRADA,
                      volteo_horizontal=False, semilla=SEMILLA, pesos="imagenet"):
    import tensorflow as tf
    from tensorflow.keras.applications.resnet50 import preprocess_input

    if numero_clases < 2:
        raise ValueError("El clasificador necesita al menos dos clases.")
    tf.keras.utils.set_random_seed(semilla)
    entrada = tf.keras.Input(shape=(*tamano, 3), dtype="float32", name="imagen_rgb")
    x = entrada
    if volteo_horizontal:
        x = tf.keras.layers.RandomFlip("horizontal", seed=semilla, name="aumento_train")(x)
    x = tf.keras.layers.Lambda(preprocess_input, name="preprocesamiento_resnet50")(x)
    modelo_base = tf.keras.applications.ResNet50(
        input_shape=(*tamano, 3), include_top=False, weights=pesos, pooling="avg",
    )
    modelo_base.trainable = False
    x = modelo_base(x, training=False)
    x = tf.keras.layers.Dropout(0.35, seed=semilla)(x)
    salida = tf.keras.layers.Dense(
        numero_clases, activation="softmax", name="clase",
        kernel_initializer=tf.keras.initializers.GlorotUniform(seed=semilla),
    )(x)
    modelo = tf.keras.Model(entrada, salida, name="resnet50_palta")
    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy", metrics=["accuracy"],
    )
    return modelo

## 5. Entrenamiento controlado

Se conservan: lote de 8, Adam con tasa 0,001, hasta 35 épocas, Early Stopping
sobre `val_loss` con paciencia 7 y reducción de tasa con paciencia 3. Los pesos
por clase se calculan solo con train y después de las exclusiones.

El tamaño y augmentation son las únicas diferencias entre B0, V1 y V2. Los
archivos elegibles, semilla, lote, cabeza y criterio de parada se mantienen iguales.
Cada ejecución crea su propia carpeta y no sobrescribe resultados anteriores.
Se guardan configuración, listas de archivos, pesos e historial. La comparación
usa validation; test permanece reservado.

Para B0 y V1 se calcula una sola vez la salida determinista del extractor congelado en memoria. La cabeza conserva Dropout y Dense, comparte sus capas con el modelo completo y se comprueba su equivalencia numérica antes de entrenar. El camino de augmentation está conservado como antecedente, pero no está habilitado para este dataset. La CPU es utilizable,
pero una entrada mayor aumenta el coste. No se presentan tiempos estimados ni
métricas hasta ejecutar con el dataset local.

In [6]:
def entrenar_resnet50(nombre_ejecucion, epocas=35, tamano=TAMANO_ENTRADA,
                      volteo_horizontal=False, aprobar_exclusiones=False,
                      semilla=SEMILLA, condicion="B0"):
    if (not nombre_ejecucion or nombre_ejecucion in (".", "..")
            or Path(nombre_ejecucion).name != nombre_ejecucion
            or any(c in nombre_ejecucion for c in '<>:"/\\|?*')):
        raise ValueError("Use un nombre simple para la carpeta de resultados.")
    if not isinstance(epocas, int) or epocas < 1:
        raise ValueError("El numero de epocas debe ser un entero positivo.")
    tamano = tuple(map(int, tamano))
    if len(tamano) != 2 or min(tamano) < 32:
        raise ValueError("Revise el alto y el ancho de entrada.")
    if condicion not in EXPERIMENTOS:
        raise ValueError("Condicion desconocida.")
    if volteo_horizontal:
        raise ValueError("V2 bloqueada: el volteo invierte el orden de capturas que debe conservarse.")
    esperado = EXPERIMENTOS[condicion]
    if tamano != esperado["tamano"] or bool(volteo_horizontal) != esperado["volteo_horizontal"]:
        raise ValueError("Los parametros no corresponden a la condicion seleccionada.")
    carpeta = SALIDA / nombre_ejecucion
    if carpeta.exists():
        raise FileExistsError("La ejecucion ya existe. Cambie REPETICION sin borrar resultados.")
    dataset = resolver_dataset(RAIZ, RUTA_DATASET_RELATIVA)
    auditoria = verificar_auditoria(dataset, aprobar_exclusiones=aprobar_exclusiones)
    dataset, tabla_train, tabla_val, clases = preparar_datos(auditoria["exclusiones"])

    import tensorflow as tf
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(semilla)
    tf.config.experimental.enable_op_determinism()
    pesos_clases = calcular_pesos(tabla_train, clases)
    protocolo = {
        "arquitectura": "ResNet50 congelada + GAP + Dropout(0.35) + Dense softmax",
        "pesos_base": "imagenet", "extractor_determinista_en_memoria": True, "clases": clases,
        "sha256_dataset": auditoria["sha256_dataset"],
        "sha256_manifiesto": auditoria["sha256_manifiesto"],
        "semilla": int(semilla), "tamano_lote": int(TAMANO_LOTE),
        "epocas_maximas": epocas, "optimizer": "Adam", "learning_rate": 0.001,
        "loss": "sparse_categorical_crossentropy", "monitor": "val_loss",
        "early_stopping_patience": 7, "reduce_lr_patience": 3, "reduce_lr_factor": 0.2,
        "pesos_clases": pesos_clases,
    }
    firma_protocolo = hashlib.sha256(
        json.dumps(protocolo, sort_keys=True).encode()
    ).hexdigest()
    configuracion = {
        "condicion": condicion, "nombre_ejecucion": nombre_ejecucion,
        "tamano_entrada": list(tamano), "volteo_horizontal": bool(volteo_horizontal),
        "protocolo_comun": protocolo, "firma_protocolo": firma_protocolo,
        "exclusiones": auditoria["exclusiones"],
        "n_train": len(tabla_train), "n_validation": len(tabla_val),
        "preprocesamiento": "RGB float32 0-255; preprocess_input dentro del modelo",
        "versiones": {p: importlib.metadata.version(p)
                      for p in ("numpy", "pandas", "Pillow", "tensorflow", "scikit-learn")},
        "python": platform.python_version(),
        "estrategia": "caracteristicas congeladas en memoria" if not volteo_horizontal else "pipeline con augmentation por epoca",
    }
    carpeta.mkdir(parents=True, exist_ok=False)
    (carpeta / "configuracion.json").write_text(
        json.dumps(configuracion, ensure_ascii=False, indent=2), encoding="utf-8")
    (carpeta / "auditoria.json").write_text(json.dumps(auditoria, indent=2), encoding="utf-8")
    tabla_train.to_csv(carpeta / "archivos_train.csv", index=False)
    tabla_val.to_csv(carpeta / "archivos_validation.csv", index=False)
    estado = carpeta / "estado.json"
    estado.write_text(json.dumps({"estado": "iniciado"}), encoding="utf-8")
    inicio = time.perf_counter()
    try:
        datos_train = crear_dataset_tf(
            tabla_train, dataset, tamano=tamano, tamano_lote=TAMANO_LOTE,
            entrenamiento=True, semilla=semilla)
        datos_val = crear_dataset_tf(
            tabla_val, dataset, tamano=tamano, tamano_lote=TAMANO_LOTE,
            entrenamiento=False, semilla=semilla)
        modelo = construir_modelo(len(clases), tamano, volteo_horizontal, semilla)
        # Con extractor congelado y sin augmentation, calcularlo una vez es
        # equivalente a repetirlo por época. Dropout y Dense siguen entrenándose.
        inicio_extraccion = time.perf_counter()
        modelo_fit = modelo
        if not volteo_horizontal:
            base = next(capa for capa in modelo.layers if isinstance(capa, tf.keras.Model))
            extractor = tf.keras.Model(modelo.input, modelo.layers[-2].input)
            datos_ordenados = crear_dataset_tf(tabla_train, dataset, tamano=tamano,
                tamano_lote=TAMANO_LOTE, entrenamiento=False, semilla=semilla)
            print("Extraccion real de train:", len(tabla_train), flush=True)
            class Progreso(tf.keras.callbacks.Callback):
                def on_predict_batch_end(self, batch, logs=None):
                    if batch % 10 == 0:
                        print(f"Lote de extraccion {batch + 1}", flush=True)
            caracteristicas_train = extractor.predict(datos_ordenados, verbose=2, callbacks=[Progreso()])
            print("Extraccion real de validation:", len(tabla_val), flush=True)
            caracteristicas_val = extractor.predict(datos_val, verbose=2, callbacks=[Progreso()])
            if not (np.isfinite(caracteristicas_train).all() and np.isfinite(caracteristicas_val).all()):
                raise RuntimeError("Caracteristicas no finitas")
            entrada_cabeza = tf.keras.Input(shape=(int(base.output.shape[-1]),))
            salida_cabeza = modelo.layers[-1](modelo.layers[-2](entrada_cabeza))
            modelo_fit = tf.keras.Model(entrada_cabeza, salida_cabeza)
            modelo_fit.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                loss="sparse_categorical_crossentropy", metrics=["accuracy"])
            def lotes(vectores, etiquetas, barajar):
                ds = tf.data.Dataset.from_tensor_slices((vectores, etiquetas))
                if barajar:
                    ds = ds.shuffle(len(vectores), seed=semilla, reshuffle_each_iteration=True)
                opciones = tf.data.Options()
                opciones.threading.private_threadpool_size = 2
                return ds.with_options(opciones).batch(TAMANO_LOTE).prefetch(1)
            datos_train = lotes(caracteristicas_train, tabla_train.etiqueta.to_numpy(), True)
            datos_val = lotes(caracteristicas_val, tabla_val.etiqueta.to_numpy(), False)
            # Verificar que la composición reproduce al modelo completo antes de fit.
            lote_rgb, _ = next(iter(datos_ordenados))
            np.testing.assert_allclose(modelo(lote_rgb, training=False).numpy(),
                modelo_fit(caracteristicas_train[:len(lote_rgb)], training=False).numpy(),
                rtol=1e-4, atol=1e-5)
        segundos_extraccion = time.perf_counter() - inicio_extraccion
        inicio_cabeza = time.perf_counter()
        checkpoint = carpeta / "mejor_cabeza.weights.h5"
        callbacks = [
            tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True),
            tf.keras.callbacks.ModelCheckpoint(str(checkpoint), monitor="val_loss",
                                               save_best_only=True, save_weights_only=True),
            tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3),
            tf.keras.callbacks.CSVLogger(str(carpeta / "historial.csv")),
            tf.keras.callbacks.TerminateOnNaN(),
        ]
        historial = modelo_fit.fit(datos_train, validation_data=datos_val, epochs=epocas,
                               class_weight=pesos_clases, callbacks=callbacks)
        if not all(np.isfinite(v).all() for v in historial.history.values()):
            raise RuntimeError("Se encontraron metricas no finitas durante el entrenamiento.")
        modelo_fit.load_weights(str(checkpoint))
        pd.DataFrame(historial.history).to_csv(carpeta / "historial.csv", index=False)
        estado.write_text(json.dumps({"estado": "entrenamiento_completo",
                                      "segundos": time.perf_counter() - inicio,
                                      "segundos_extraccion": segundos_extraccion,
                                      "segundos_cabeza": time.perf_counter() - inicio_cabeza,
                                      "epocas": len(historial.history["loss"]),
                                      "mejor_epoca_val_loss": int(np.argmin(historial.history["val_loss"])) + 1,
                                      "memoria_pico_proceso_mib": __import__("psutil").Process().memory_info().peak_wset / 2**20}), encoding="utf-8")
    except (Exception, KeyboardInterrupt) as error:
        estado.write_text(json.dumps({"estado": "incompleto", "detalle": str(error), "traceback": __import__("traceback").format_exc()}), encoding="utf-8")
        raise
    return {"modelo": modelo, "historial": historial, "carpeta": carpeta,
            "clases": clases, "tamano_entrada": tamano, "semilla": semilla,
            "sha256_dataset": auditoria["sha256_dataset"],
            "auditoria": auditoria, "configuracion": configuracion,
            "tabla_val": tabla_val, "dataset": dataset}

## 6. Curvas de entrenamiento

Las curvas se calculan a partir de una ejecución real. No hay métricas ni resultados
simulados en este notebook.


In [7]:
def graficar_curvas(historial, carpeta=None):
    import matplotlib.pyplot as plt

    figuras = []
    valores = historial.history
    for metrica, titulo in (("loss", "Perdida"), ("accuracy", "Accuracy")):
        figura, eje = plt.subplots(figsize=(7, 4))
        epocas = np.arange(1, len(valores[metrica]) + 1)
        eje.plot(epocas, valores[metrica], label="Train")
        eje.plot(epocas, valores[f"val_{metrica}"], label="Validation")
        eje.set(xlabel="Epoca", ylabel=titulo, title=titulo)
        eje.grid(alpha=0.3)
        eje.legend()
        figura.tight_layout()
        if carpeta is not None:
            figura.savefig(Path(carpeta) / f"curva_{metrica}.png", dpi=150)
        plt.show()
        figuras.append(figura)
        plt.close(figura)
    return figuras

## 7. Evaluación de validation y test reservado

El informe compara B0 y las variantes sobre **validation**, con macro-F1, balanced
accuracy, precisión, recall y F1 por clase. Se guardan las predicciones por archivo
para comprobar los errores. Los valores se generan únicamente con una ejecución
real; no hay métricas de ejemplo en los CSV.

La función `evaluar_test` se conserva para la evaluación final, pero no se llama
en este notebook. Test no participa en la elección de variantes. El umbral no se
optimiza: se utiliza `argmax` de la salida softmax. Las exclusiones aprobadas también
se aplican a test cuando corresponda evaluarlo.

In [8]:
def evaluar_particion(experimento, particion="validation"):
    import matplotlib.pyplot as plt
    from sklearn.metrics import (
        classification_report, confusion_matrix, ConfusionMatrixDisplay,
        balanced_accuracy_score, f1_score, accuracy_score,
    )
    if particion not in {"validation", "test"}:
        raise ValueError("Use validation o test.")
    dataset = resolver_dataset(RAIZ, RUTA_DATASET_RELATIVA)
    auditoria = verificar_auditoria(dataset, aprobar_exclusiones=True)
    if auditoria != experimento["auditoria"]:
        raise RuntimeError("El dataset o las exclusiones cambiaron respecto al entrenamiento.")
    clases = experimento["clases"]
    tabla = listar_imagenes(dataset, particion, clases=clases)
    tabla = tabla.loc[~tabla["archivo"].isin(auditoria["exclusiones"])].reset_index(drop=True)
    if tabla.empty or set(tabla["clase"]) != set(clases):
        raise ValueError("La evaluacion debe conservar ambas clases.")
    datos = crear_dataset_tf(
        tabla, dataset, tamano=experimento["tamano_entrada"],
        tamano_lote=experimento["configuracion"]["protocolo_comun"]["tamano_lote"],
        entrenamiento=False, semilla=experimento["semilla"],
    )
    probabilidades = np.asarray(experimento["modelo"].predict(datos, verbose=0))
    if probabilidades.shape != (len(tabla), len(clases)) or not np.isfinite(probabilidades).all():
        raise RuntimeError("Forma o valores de prediccion invalidos.")
    predicciones = np.argmax(probabilidades, axis=1)
    verdaderos = tabla["etiqueta"].to_numpy(dtype=np.int32)
    indices = np.arange(len(clases))
    reporte = classification_report(verdaderos, predicciones, labels=indices,
                                    target_names=clases, output_dict=True, zero_division=0)
    configuracion = experimento["configuracion"]
    metricas = {
        "estado": "completo", "particion": particion, "n_imagenes": len(tabla),
        "condicion": configuracion["condicion"],
        "nombre_ejecucion": configuracion["nombre_ejecucion"],
        "firma_protocolo": configuracion["firma_protocolo"],
        "accuracy": float(accuracy_score(verdaderos, predicciones)),
        "macro_f1": float(f1_score(verdaderos, predicciones, labels=indices,
                                    average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(verdaderos, predicciones)),
        "recall_picadura": float(reporte["PICADURA"]["recall"]),
        "recall_russet": float(reporte["RUSSET"]["recall"]),
    }
    carpeta = experimento["carpeta"]
    pd.DataFrame(reporte).transpose().to_csv(carpeta / f"metricas_por_clase_{particion}.csv")
    resultados = tabla[["archivo", "clase", "etiqueta"]].copy()
    resultados["prediccion"] = [clases[indice] for indice in predicciones]
    for indice, clase in enumerate(clases):
        resultados[f"probabilidad_{clase}"] = probabilidades[:, indice]
    resultados.to_csv(carpeta / f"predicciones_{particion}.csv", index=False)
    matriz = confusion_matrix(verdaderos, predicciones, labels=indices)
    pd.DataFrame(matriz, index=clases, columns=clases).to_csv(carpeta / f"matriz_confusion_{particion}.csv")
    grafico = ConfusionMatrixDisplay(matriz, display_labels=clases).plot()
    grafico.ax_.set_title(f"{configuracion['condicion']} - {particion}")
    grafico.figure_.tight_layout()
    grafico.figure_.savefig(carpeta / f"matriz_confusion_{particion}.png", dpi=150)
    plt.show()
    plt.close(grafico.figure_)
    (carpeta / f"metricas_{particion}.json").write_text(
        json.dumps(metricas, ensure_ascii=False, indent=2), encoding="utf-8")
    print(pd.DataFrame(reporte).transpose().round(4))
    return metricas, resultados


def evaluar_validacion(experimento):
    return evaluar_particion(experimento, "validation")


def evaluar_test(experimento):
    return evaluar_particion(experimento, "test")

## 8. Ejecución paso a paso

No es necesario repetir el EDA ni Feature Engineering si los archivos no cambiaron.
La auditoría vuelve a comprobarlos antes de cada experimento.

1. Seleccionar el kernel `palta`, verificar `RUTA_DATASET_RELATIVA = "../DATASET_PALTA"`
   y ejecutar las secciones 1 a 7. No entrenan.
2. Ejecutar la celda de propuesta que sigue. Revisar los cuatro archivos excluidos.
   La propuesta usa reportes existentes: todavía no comprueba el dataset local.
3. En la celda de controles, dejar `ENTRENAR = False`; aprobar las exclusiones
   con `APROBAR_EXCLUSIONES = True` solo después de revisarlas y activar
   `REVISAR_INTEGRIDAD = True`. Ejecutar esa celda. Debe indicar integridad correcta.
4. Para B0, fijar `CONDICION = "B0"` y `ENTRENAR = True` en esa misma celda. Ejecutarla.
   Entrena, evalúa validation y guarda curvas, predicciones y métricas.
5. Poner `ENTRENAR = False` inmediatamente después. Para V1, cambiar a `"V1"`,
   volver a activar `ENTRENAR = True` y ejecutar la misma celda. V2 está bloqueada por incompatibilidad con el orden requerido de las capturas.
6. Ejecutar la comparación de la sección 9 y completar la tabla del informe
   editable con las métricas reales. Guardar el notebook con sus salidas.

**No usar Run All con `ENTRENAR = True` sin revisar qué condición se ejecutará.**
Con los controles iniciales en False, Run All es una revisión sin entrenamiento.

`REPETICION = "01"` identifica la primera comparación. Si una ejecución ya existe,
no se sobrescribe: cambiar el sufijo a `"02"` para la nueva serie y usarlo en la
comparación. Para repetir sin cambiar condiciones, mantener semilla y parámetros.
Una sola semilla produce una comparación preliminar, no significancia estadística.

In [9]:
# Propuesta basada en los reportes guardados, sin abrir TIFF.
resumen_eda, inventario_eda, propuesta_exclusiones = leer_auditoria()
print("Archivos propuestos para excluir solo del experimento:")
display(inventario_eda.loc[inventario_eda["archivo"].isin(propuesta_exclusiones),
                           ["archivo", "particion", "clase"]])
conteos_originales = inventario_eda.groupby("particion").size()
conteos_elegibles = inventario_eda.loc[
    ~inventario_eda["archivo"].isin(propuesta_exclusiones)
].groupby("particion").size()
display(pd.DataFrame({"original": conteos_originales, "elegible_propuesto": conteos_elegibles})
        .reindex(["train", "validation", "test"]))
print("No se modificaron imagenes. Falta validar integridad local y aprobar la propuesta.")

Archivos propuestos para excluir solo del experimento:


,archivo,particion,clase
111,test/RUSSET/704F3E54-250612-145719-5.RGB.tiff,test,RUSSET
136,train/PICADURA/704F3E54-250505-153013-0.RGB.tiff,train,PICADURA
169,train/PICADURA/704F3E54-250612-145719-5.RGB.tiff,train,PICADURA
807,validation/PICADURA/defecto.RGB.tiff,validation,PICADURA


,original,elegible_propuesto
particion,,
train,574,572
validation,128,127
test,127,126


No se modificaron imagenes. Falta validar integridad local y aprobar la propuesta.


In [10]:
# Cambiar una condicion por vez.
CONDICION = "B0"                # B0, V1 o V2.
REPETICION = "01"
EPOCAS = 35
APROBAR_EXCLUSIONES = False
REVISAR_INTEGRIDAD = False
ENTRENAR = False

if CONDICION not in EXPERIMENTOS:
    raise ValueError("Seleccione B0, V1 o V2.")
nombre_ejecucion = f"{CONDICION}_s{SEMILLA}_{REPETICION}"
if ENTRENAR:
    parametros = EXPERIMENTOS[CONDICION]
    experimento = entrenar_resnet50(
        nombre_ejecucion, epocas=EPOCAS, tamano=parametros["tamano"],
        volteo_horizontal=parametros["volteo_horizontal"],
        aprobar_exclusiones=APROBAR_EXCLUSIONES, semilla=SEMILLA, condicion=CONDICION,
    )
    metricas, predicciones = evaluar_validacion(experimento)
    graficar_curvas(experimento["historial"], experimento["carpeta"])
    print("Resultados:", experimento["carpeta"].relative_to(RAIZ).as_posix())
    print("Finalizado. Vuelva a dejar ENTRENAR = False antes de guardar.")
elif REVISAR_INTEGRIDAD:
    dataset = resolver_dataset(RAIZ, RUTA_DATASET_RELATIVA)
    auditoria = verificar_auditoria(dataset, aprobar_exclusiones=APROBAR_EXCLUSIONES)
    print("Integridad correcta. Archivos elegibles:", auditoria["archivos_elegibles"])
    print("No se entreno ningun modelo.")
else:
    print("Modo seguro. Seleccion:", nombre_ejecucion)
    print("Revise las exclusiones y siga los pasos de la seccion 8.")

Modo seguro. Seleccion: B0_s42_01
Revise las exclusiones y siga los pasos de la seccion 8.


## 9. Comparación y actualización del informe

Esta celda lee únicamente resultados completos de validation. Un experimento no
realizado permanece como pendiente; no se le asigna cero. La firma del protocolo
impide mezclar condiciones con diferentes archivos, semilla, lote o hiperparámetros.
Se completaron B0 y V1. V2 figura como no ejecutada con su causa explícita.

Usar `comparacion_validation.csv`, las curvas y las matrices de confusión para
completar `2. informe/Informe.docx`. Justificar cualquier mejora o empeoramiento sin
suponer que más resolución o augmentation siempre ayudan. No presentar test como
validación ni afirmar resultados de clasificación mientras las filas estén pendientes.

In [11]:
def comparar_experimentos(semilla=SEMILLA, repeticion="01"):
    filas, firmas = [], set()
    for condicion in EXPERIMENTOS:
        nombre = f"{condicion}_s{semilla}_{repeticion}"
        carpeta = SALIDA / nombre
        ruta = carpeta / "metricas_validation.json"
        fila = {"condicion": condicion, "ejecucion": nombre, "estado": "pendiente"}
        if condicion == "V2":
            fila.update(estado="no_ejecutada", motivo="Volteo incompatible con el orden requerido de capturas")
        if ruta.is_file():
            metricas = json.loads(ruta.read_text(encoding="utf-8"))
            configuracion = json.loads((carpeta / "configuracion.json").read_text(encoding="utf-8"))
            if (metricas.get("estado") != "completo" or metricas.get("particion") != "validation"
                    or metricas.get("condicion") != condicion):
                raise ValueError(f"Resultado no valido: {nombre}")
            if (metricas["firma_protocolo"] != configuracion["firma_protocolo"]
                    or configuracion["protocolo_comun"]["semilla"] != semilla
                    or configuracion["tamano_entrada"] != list(EXPERIMENTOS[condicion]["tamano"])
                    or configuracion["volteo_horizontal"] != EXPERIMENTOS[condicion]["volteo_horizontal"]):
                raise ValueError(f"Configuracion inconsistente: {nombre}")
            firmas.add(metricas["firma_protocolo"])
            fila["estado"] = "completo"
            for clave in ("accuracy", "macro_f1", "balanced_accuracy", "recall_picadura", "recall_russet"):
                valor = float(metricas[clave])
                if not np.isfinite(valor) or not 0 <= valor <= 1:
                    raise ValueError(f"Metrica invalida: {nombre}/{clave}")
                fila[clave] = valor
        filas.append(fila)
    if len(firmas) > 1:
        raise RuntimeError("No comparar: cambiaron el dataset, semilla o protocolo comun.")
    tabla = pd.DataFrame(filas)
    if "macro_f1" in tabla and tabla.loc[0, "estado"] == "completo":
        tabla["delta_macro_f1_vs_B0"] = tabla["macro_f1"] - tabla.loc[0, "macro_f1"]
    if firmas:
        SALIDA.mkdir(parents=True, exist_ok=True)
        tabla.to_csv(SALIDA / "comparacion_validation.csv", index=False)
    return tabla


comparacion = comparar_experimentos(semilla=SEMILLA, repeticion=REPETICION)
display(comparacion.round(4))
if (comparacion["estado"] == "completo").sum() < 2 or comparacion.loc[0, "estado"] != "completo":
    print("Falta completar B0 y al menos una variante antes de concluir la comparacion.")

,condicion,ejecucion,estado,accuracy,macro_f1,balanced_accuracy,recall_picadura,recall_russet,motivo,delta_macro_f1_vs_B0
0,B0,B0_s42_01,completo,0.8425,0.6604,0.6384,0.9434,0.3333,NaN,0.0000
1,V1,V1_s42_01,completo,0.8583,0.6598,0.6287,0.9717,0.2857,NaN,-0.0006
2,V2,V2_s42_01,no_ejecutada,NaN,NaN,NaN,NaN,NaN,Volteo incompatible con el orden requerido de ...,NaN


### Archivos que se obtienen al ejecutar

En `7. resultados/entrenamiento/B0_s42_01/` (y su equivalente para cada variante):
`configuracion.json`, `auditoria.json`, `archivos_train.csv`, `archivos_validation.csv`,
`historial.csv`, `mejor_cabeza.weights.h5`, `estado.json`, `metricas_validation.json`,
`metricas_por_clase_validation.csv`, `predicciones_validation.csv`,
`matriz_confusion_validation.png`, `curva_loss.png` y `curva_accuracy.png`.

Los archivos `.h5` siguen excluidos por el `.gitignore` existente. Revisar siempre
los resultados antes de publicarlos. No subir el dataset ni los entornos virtuales.

### Cierre

El informe se actualiza con los resultados reales de B0 y V1 y documenta por qué V2 no se ejecuta. No se cambiaron los notebooks anteriores,
los reportes originales, las particiones ni las funciones de lectura compartidas.

## Ejecución verificada

B0 y V1 se entrenaron realmente usando todas las imágenes elegibles de train y se evaluaron en validation. Ambas corridas terminaron en 15 épocas y recuperaron la época 8 por menor val_loss. Las funciones de este notebook se ejecutaron con el intérprete del entorno palta; el notebook completo se ejecutó después en modo seguro para guardar la comparación.

Se usaron 8 hilos intra-op y 2 inter-op (`TF_NUM_INTRAOP_THREADS=8`, `TF_NUM_INTEROP_THREADS=2`) y un proceso independiente por condición para medir memoria. Los valores por defecto de estos hilos pueden afectar tiempos al repetir en VS Code. Los pesos ImageNet se conservan únicamente en `.cache/keras`, excluida de Git. Los checkpoints de esta ejecución contienen la cabeza; para reconstruir el modelo se deben cargar en una cabeza con las mismas capas, sobre la ResNet50 ImageNet congelada.

B0: macro-F1 0,6604; V1: macro-F1 0,6598. La accuracy superior de V1 no implica una mejora en RUSSET: su recall baja de 7/21 a 6/21. La comparación es exploratoria, con una sola semilla. No se evaluó test. V2 quedó bloqueada tras confirmar que el orden horizontal de las capturas debe conservarse.
